# 1. Introduction and feature engineering objectives

Each row represents `(wallet, as_of)`. Every registered value is calculated only from validated events satisfying `block_time < as_of`. These features describe observed behavior; none independently establishes wash trading.

## 2. Wallet-level prediction unit and reproducible setup

The unit is one wallet observed at one explicit UTC cutoff. Wallet and cutoff remain metadata; only the 29 registered aggregate fields are candidate attributes.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (path / 'pyproject.toml').is_file())
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('MPLCONFIGDIR', '/tmp/solana-risk-matplotlib')

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from solana_risk.features.build import (
    FEATURE_REGISTRY, MODEL_FEATURE_COLUMNS, build_wallet_snapshots,
    feature_registry_records, load_default_cleaned_events, parse_as_of,
)

sns.set_theme(style='whitegrid')
CUTOFFS = ['2025-09-01T00:00:00Z', '2025-10-01T00:00:00Z', '2025-12-01T00:00:00Z']

## 3. Load the checksum-verified Phase 2 events

In [ ]:
events = load_default_cleaned_events()
{'events': len(events), 'wallets': events['wallet'].nunique(), 'start_utc': events['block_time'].min(), 'end_utc': events['block_time'].max()}

## 4. Explicit scoring timestamps and strict temporal cutoff

The cutoff is excluded. A lookback window is closed on its lower bound and open on its upper bound: `as_of - window <= block_time < as_of`. A wallet with no prior event receives no row, because unavailable history must not be rewritten as zero activity.

In [ ]:
cutoffs = [parse_as_of(value) for value in CUTOFFS]
pd.DataFrame({'as_of': cutoffs, 'strictly_prior_events': [events['block_time'].lt(value).sum() for value in cutoffs], 'excluded_events': [events['block_time'].ge(value).sum() for value in cutoffs]})

## 5. Build deterministic wallet snapshots with the reusable module

In [ ]:
snapshots = build_wallet_snapshots(events, cutoffs)
assert list(snapshots.columns) == ['wallet', 'as_of', *MODEL_FEATURE_COLUMNS]
snapshots.groupby('as_of').size().rename('wallet_snapshots').to_frame()

## 6. Feature registry and lineage

In [ ]:
registry = pd.DataFrame(feature_registry_records())
assert registry['name'].is_unique and len(registry) == len(MODEL_FEATURE_COLUMNS)
display(registry[['name', 'category', 'calculation', 'units', 'missingness_policy', 'lineage']])

## 7. Wallet-activity features

In [ ]:
activity = ['historical_event_count', 'unique_transaction_count', 'active_day_count', 'events_per_active_day', 'observed_activity_duration_days']
display(snapshots[activity].describe().T)

## 8. Trading-volume features and distribution analysis

Summary quantiles and a log-log histogram expose the heavy tail without fitting a transform or deleting genuine extreme values.

In [ ]:
volume = ['total_usd_volume', 'mean_usd_volume', 'median_usd_volume', 'max_usd_volume', 'std_usd_volume']
display(snapshots[volume].describe(percentiles=[.5, .9, .95, .99]).T)
sns.histplot(snapshots['total_usd_volume'], bins=50, log_scale=(True, True)); plt.title('Point-in-time total USD volume (log-log)'); plt.show()

## 9. Temporal gaps and lookback-window features

Window-count distributions are dominated by zero at these cutoffs, consistent with sparse cross-month wallet overlap rather than proof of inactivity outside the observed data.

In [ ]:
temporal = ['time_since_last_event_hours', 'inter_event_mean_seconds', 'inter_event_median_seconds', 'inter_event_std_seconds', 'events_last_24h', 'events_last_7d', 'events_last_30d']
display(snapshots[temporal].describe().T)
snapshots.groupby('as_of')[['events_last_24h', 'events_last_7d', 'events_last_30d']].mean().plot(marker='o', title='Mean event-window counts by cutoff'); plt.show()

## 10. Token and DEX diversity/concentration features

Diversity and concentration are interpreted with historical event counts: a concentration of one from one event is limited evidence, not a behavioral verdict.

In [ ]:
market = ['unique_token_count', 'unique_token_pair_count', 'unique_dex_count', 'token_concentration', 'dex_concentration']
display(snapshots[market].describe().T)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(snapshots['token_concentration'], bins=25, ax=axes[0]); axes[0].set_title('Token concentration')
sns.histplot(snapshots['dex_concentration'], bins=25, ax=axes[1]); axes[1].set_title('DEX concentration')
plt.tight_layout(); plt.show()

## 11. Fee, transaction, and observable-history quality features

In [ ]:
quality = ['missing_fee_proportion', 'multi_instruction_transaction_count', 'single_event_history', 'insufficient_history_7d', 'insufficient_history_30d', 'observable_history_days', 'history_coverage_30d_fraction']
display(snapshots[quality].describe().T)
display(snapshots[['single_event_history', 'insufficient_history_7d', 'insufficient_history_30d']].mean().rename('snapshot_proportion').to_frame())

## 12. Feature missingness and explicit unavailable history

In [ ]:
feature_missing = snapshots[list(MODEL_FEATURE_COLUMNS)].isna().mean().sort_values(ascending=False)
display(feature_missing[feature_missing.gt(0)].rename('missing_proportion').to_frame())
assert snapshots.loc[snapshots['single_event_history'], ['inter_event_mean_seconds', 'inter_event_median_seconds', 'inter_event_std_seconds']].isna().all().all()

## 13. Descriptive feature relationships (no target)

In [ ]:
correlation_columns = ['historical_event_count', 'active_day_count', 'total_usd_volume', 'time_since_last_event_hours', 'events_last_30d', 'unique_token_count', 'unique_dex_count', 'token_concentration', 'dex_concentration', 'observable_history_days']
plt.figure(figsize=(10, 8))
sns.heatmap(snapshots[correlation_columns].corr(method='spearman'), cmap='vlag', center=0, vmin=-1, vmax=1);
plt.title('Spearman feature relationships (descriptive only)'); plt.tight_layout(); plt.show()

## 14. Explicit leakage demonstration

In [ ]:
demonstration_cutoff = cutoffs[1]
from_all_events = build_wallet_snapshots(events, [demonstration_cutoff])
prior_only = events.loc[events['block_time'].lt(demonstration_cutoff)].copy()
from_prefiltered_events = build_wallet_snapshots(prior_only, [demonstration_cutoff])
pd.testing.assert_frame_equal(from_all_events, from_prefiltered_events)
assert not events.loc[events['block_time'].ge(demonstration_cutoff)].empty
{'cutoff': demonstration_cutoff, 'excluded_at_or_after_cutoff': int(events['block_time'].ge(demonstration_cutoff).sum()), 'snapshots_identical_after_explicit_prefilter': True}

## 15. Feature limitations, interpretation, and next steps

The module creates 29 deterministic behavioral and data-quality features while keeping wallet and cutoff as metadata. It recomputes history from validated events and never uses supplied previous-event, gap, or source-week fields. Single-event and short-history flags prevent unavailable temporal evidence from appearing complete. The source has sparse cross-month wallet coverage, no verified wash-trading labels, no counterparty edges, and unresolved provenance/licensing. Consequently no supervised target, label-dependent feature selection, scaling, resampling, model, metric, graph claim, or risk score is created. Phase 4 may start only after explicit approval and an independently verified time-aware label source passes the documented gate.